In [1]:
!python -V

Python 3.11.9


In [2]:
!pip list

Package                 Version
----------------------- -----------
asttokens               3.0.1
catboost                1.2.10
colorama                0.4.6
comm                    0.2.3
contourpy               1.3.3
cycler                  0.12.1
debugpy                 1.8.20
decorator               5.2.1
executing               2.2.1
fonttools               4.62.1
graphviz                0.21
ipykernel               7.2.0
ipython                 9.13.0
ipython_pygments_lexers 1.1.1
jedi                    0.20.0
joblib                  1.5.3
jupyter_client          8.8.0
jupyter_core            5.9.1
kiwisolver              1.5.0
matplotlib              3.10.9
matplotlib-inline       0.2.1
narwhals                2.20.0
nest-asyncio            1.6.0
numpy                   2.4.4
packaging               26.2
pandas                  3.0.2
parso                   0.8.7
pillow                  12.2.0
pip                     26.1.1
platformdirs            4.9.6
plotly                  

In [9]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor, Pool

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.5f}")

print("Python:", sys.version)
print("Working directory:", os.getcwd())

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Working directory: c:\MyPythonProject\stage_1_solution


In [10]:
df = pd.read_csv("./train.csv")

ID_COL = "new_id"
MONTH_COL = "Месяц"
TARGET = "РТО"

cat_cols = [
    "Дата открытия, категориальный",
    "Торговая площадь, категориальный",
    "Населенный пункт",
    "Регион",
]

num_static_cols = [
    "Численность населения",
    "Количество домохозяйств",
    "Трафик пеший, в час",
    "Трафик авто, в час",
    "Маркетплейсы, доставки, постаматы (100 м)",
    "Медицинские уч. и аптеки (300 м)",
    "Школы (300 м)",
    "Остановки (300 м)",
    "Продуктовые магазины (500 м)",
    "Пятерочки (500 м)",
    "Количество касс",
    "Флаг алкогольной лицензии",
]

cat_cols = [c for c in cat_cols if c in df.columns]
num_static_cols = [c for c in num_static_cols if c in df.columns]

for c in cat_cols:
    df[c] = df[c].astype(str)

print("Shape:", df.shape)
print("Stores:", df[ID_COL].nunique())
print("Months:", sorted(df[MONTH_COL].unique()))

display(df.head())

Shape: (206150, 19)
Stores: 20615
Months: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


,new_id,Месяц,"Дата открытия, категориальный","Торговая площадь, категориальный",Населенный пункт,Регион,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м),Количество касс,Флаг алкогольной лицензии,РТО
0,0,1,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,9588,501,79,156,0,6,0,0,2,0,10,1,"22,041,969.33000"
1,0,2,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,9588,501,79,156,0,6,0,0,2,0,10,1,"23,268,490.57000"
2,0,3,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,9588,501,79,156,0,6,0,0,2,0,10,1,"24,487,732.21000"
3,0,4,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,9588,501,79,156,0,6,0,0,2,0,10,1,"23,981,980.29000"
4,0,5,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,9588,501,79,156,0,6,0,0,2,0,10,1,"26,608,343.78000"


In [11]:
df_model = df.sort_values([ID_COL, MONTH_COL]).copy()

df_model["rto_lag_1"] = df_model.groupby(ID_COL)[TARGET].shift(1)
df_model["rto_lag_2"] = df_model.groupby(ID_COL)[TARGET].shift(2)
df_model["rto_lag_3"] = df_model.groupby(ID_COL)[TARGET].shift(3)

df_model["rto_mean_2"] = (
    df_model.groupby(ID_COL)[TARGET]
    .transform(lambda s: s.shift(1).rolling(2).mean())
)

df_model["rto_mean_3"] = (
    df_model.groupby(ID_COL)[TARGET]
    .transform(lambda s: s.shift(1).rolling(3).mean())
)

df_model["rto_std_3"] = (
    df_model.groupby(ID_COL)[TARGET]
    .transform(lambda s: s.shift(1).rolling(3).std())
)

df_model["growth_1_2"] = (
    df_model["rto_lag_1"] / df_model["rto_lag_2"].clip(lower=1e-9)
)

df_model["growth_2_3"] = (
    df_model["rto_lag_2"] / df_model["rto_lag_3"].clip(lower=1e-9)
)

df_model["growth_mean_2"] = df_model[["growth_1_2", "growth_2_3"]].mean(axis=1)

df_model_clean = df_model.dropna(
    subset=[
        "rto_lag_1",
        "rto_lag_2",
        "rto_lag_3",
        "rto_mean_2",
        "rto_mean_3",
        "rto_std_3",
        "growth_1_2",
        "growth_2_3",
        "growth_mean_2",
    ]
).copy()

print("df_model_clean:", df_model_clean.shape)
print("Months:", sorted(df_model_clean[MONTH_COL].unique()))

df_model_clean: (144305, 28)
Months: [np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


In [12]:
numeric_features = [
    MONTH_COL,
    "rto_lag_1",
    "rto_lag_2",
    "rto_lag_3",
    "rto_mean_2",
    "rto_mean_3",
    "rto_std_3",
    "growth_1_2",
    "growth_2_3",
    "growth_mean_2",
] + num_static_cols

categorical_features = cat_cols.copy()

numeric_features = [c for c in numeric_features if c in df_model_clean.columns]
categorical_features = [c for c in categorical_features if c in df_model_clean.columns]

features = numeric_features + categorical_features

print("Features:", len(features))
display(pd.DataFrame({"feature": features}))

Features: 26


,feature
0,Месяц
1,rto_lag_1
2,rto_lag_2
3,rto_lag_3
4,rto_mean_2
5,rto_mean_3
6,rto_std_3
7,growth_1_2
8,growth_2_3
9,growth_mean_2


In [13]:
train_final_df = df_model_clean[
    (df_model_clean[MONTH_COL] >= 4) &
    (df_model_clean[MONTH_COL] <= 10)
].copy()

X_train_final = train_final_df[features].copy()
y_train_final = train_final_df[TARGET].values

for c in categorical_features:
    X_train_final[c] = X_train_final[c].astype(str)

print("train_final_df:", train_final_df.shape)
print("train final months:", sorted(train_final_df[MONTH_COL].unique()))

train_final_df: (144305, 28)
train final months: [np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


In [14]:
test_11_df = df[df[MONTH_COL] == 10].copy()

test_11_df[MONTH_COL] = 11
test_11_df = test_11_df.drop(columns=[TARGET], errors="ignore")

rto_m10 = (
    df[df[MONTH_COL] == 10][[ID_COL, TARGET]]
    .rename(columns={TARGET: "rto_lag_1"})
)

rto_m9 = (
    df[df[MONTH_COL] == 9][[ID_COL, TARGET]]
    .rename(columns={TARGET: "rto_lag_2"})
)

rto_m8 = (
    df[df[MONTH_COL] == 8][[ID_COL, TARGET]]
    .rename(columns={TARGET: "rto_lag_3"})
)

test_11_df = test_11_df.merge(rto_m10, on=ID_COL, how="left")
test_11_df = test_11_df.merge(rto_m9, on=ID_COL, how="left")
test_11_df = test_11_df.merge(rto_m8, on=ID_COL, how="left")

test_11_df["rto_mean_2"] = test_11_df[["rto_lag_1", "rto_lag_2"]].mean(axis=1)
test_11_df["rto_mean_3"] = test_11_df[["rto_lag_1", "rto_lag_2", "rto_lag_3"]].mean(axis=1)
test_11_df["rto_std_3"] = test_11_df[["rto_lag_1", "rto_lag_2", "rto_lag_3"]].std(axis=1)

test_11_df["growth_1_2"] = (
    test_11_df["rto_lag_1"] / test_11_df["rto_lag_2"].clip(lower=1e-9)
)

test_11_df["growth_2_3"] = (
    test_11_df["rto_lag_2"] / test_11_df["rto_lag_3"].clip(lower=1e-9)
)

test_11_df["growth_mean_2"] = test_11_df[["growth_1_2", "growth_2_3"]].mean(axis=1)

for c in categorical_features:
    test_11_df[c] = test_11_df[c].astype(str)

X_test_11 = test_11_df[features].copy()

for c in categorical_features:
    X_test_11[c] = X_test_11[c].astype(str)

print("test_11_df:", test_11_df.shape)
print("X_test_11:", X_test_11.shape)
print("columns equal:", list(X_train_final.columns) == list(X_test_11.columns))

test_11_df: (20615, 27)
X_test_11: (20615, 26)
columns equal: True


In [16]:
base_train_final = train_final_df["rto_lag_1"].values

y_train_final_growth = np.log(
    np.clip(
        y_train_final / np.clip(base_train_final, 1e-9, None),
        1e-6,
        1e6
    )
)

cat_feature_indices_final = [
    X_train_final.columns.get_loc(c)
    for c in categorical_features
]

cat_growth_final_model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=10,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=RANDOM_STATE,
    verbose=200,
    allow_writing_files=False,
)

cat_growth_final_model.fit(
    Pool(
        X_train_final,
        y_train_final_growth,
        cat_features=cat_feature_indices_final,
    ),
    use_best_model=False,
)

0:	learn: 0.0634690	total: 47.2ms	remaining: 1m 10s
200:	learn: 0.0420465	total: 8.97s	remaining: 58s
400:	learn: 0.0403453	total: 17.5s	remaining: 47.9s
600:	learn: 0.0393205	total: 26s	remaining: 39s
800:	learn: 0.0387222	total: 34.7s	remaining: 30.3s
1000:	learn: 0.0383029	total: 43.6s	remaining: 21.7s
1200:	learn: 0.0379762	total: 52.3s	remaining: 13s
1400:	learn: 0.0377293	total: 1m 1s	remaining: 4.31s
1499:	learn: 0.0376210	total: 1m 5s	remaining: 0us


CatBoostRegressor(allow_writing_files=False, depth=7, eval_metric='MAE', iterations=1500, l2_leaf_reg=10, learning_rate=0.03, loss_function='MAE', random_seed=42, verbose=200)

In [19]:
base_test_11 = test_11_df["rto_lag_1"].values

pred_11_raw = cat_growth_final_model.predict(X_test_11)

pred_11 = base_test_11 * np.exp(pred_11_raw)
pred_11 = np.clip(pred_11, 1, None)

pred_11_df = test_11_df[[ID_COL]].copy()
pred_11_df["pred_11"] = pred_11

In [20]:
submission = pd.DataFrame()
submission["new_id"] = pred_11_df[ID_COL].values
submission["rto"] = pred_11_df["pred_11"].values

display(submission.head())

submission.to_csv("test.csv", index=False)

,new_id,rto
0,0,"27,936,847.16669"
1,1,"33,328,210.76604"
2,2,"38,349,077.91682"
3,3,"72,216,737.06002"
4,4,"44,533,518.63499"
